# ex002_PHT3D_02

In [ ]:
import pandas as pd
from IPython.display import display

comparison_rows = []
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

CASE_DIR = Path.cwd()
output = CASE_DIR / "output"
results = np.load(output / "results.npy")
headings = (output / "results_headings.txt").read_text(encoding="utf-8-sig").splitlines()
CASE_DIR = Path.cwd()
INPUT_DIR = CASE_DIR / "input_data"
OUTPUT_DIR = output
config = {
    "font.family": "Arial",
    "font.size": 12,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "mathtext.fontset": "stix",
    "savefig.dpi": 300,
}
plt.rcParams.update(config)
mf6pqc_data = np.load(OUTPUT_DIR / "results.npy")
mf6pqc_data = mf6pqc_data[-1].reshape(8, 50)
reference = np.load(INPUT_DIR / "PHT3D_02_results.npy", allow_pickle=False)
pht3d_data = np.stack([reference[name] for name in ("Ca", "Mg", "Cl", "pH", "Calcite", "Dolomite")])
labels = ["Ca", "Mg", "Cl", "pH", "Calcite", "Dolomite"]
letters = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]
data_funcs = [
    lambda mf, pht, snia: (mf[3, :] * 1000, pht[0, :] * 1000, snia[3, :] * 1000),
    lambda mf, pht, snia: (mf[2, :], pht[1, :], snia[2, :]),
    lambda mf, pht, snia: (mf[1, :] * 1000, pht[2, :] * 1000, snia[1, :] * 1000),
    lambda mf, pht, snia: (mf[0, :], pht[3, :], snia[0, :]),
    lambda mf, pht, snia: (mf[4, :] * 1000 / 1.8, pht[4, :] * 1000 / 1.8, snia[4, :] * 1000 / 1.8),
    lambda mf, pht, snia: (mf[5, :] * 1000 / 1.8, pht[5, :] * 1000 / 1.8, snia[5, :] * 1000 / 1.8),
]
timeseries = range(mf6pqc_data.shape[1])
fig, axes = plt.subplots(3, 2, figsize=(9.5, 6), constrained_layout=True)
axes = axes.flatten()
for idx, ax in enumerate(axes):
    y1, y2, y3 = data_funcs[idx](mf6pqc_data, pht3d_data, mf6pqc_data)
    comparison_rows.append(
        {
            "Variable": labels[idx],
            "RMSE": np.sqrt(
                np.mean((results[-1, headings.index(labels[idx])] - reference[labels[idx]]) ** 2)
            ),
        }
    )
    ax.plot(timeseries, y2, linestyle="--", label="PHT3D", color="#b22222")
    ax.plot(timeseries, y3, linestyle="-", label="SNIA", color="green")
    ax.grid(visible=True, which="major", axis="both", alpha=0.5, linestyle="--")
    ax.set_axisbelow(True)
    if idx % 2 == 0:
        ax.set_ylabel("mmol/L" if idx < 4 else "mmol/g")
    ax.text(
        0.95,
        0.95,
        f"{letters[idx]} {labels[idx]}",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=13,
    )
    if idx == 1:
        ax.legend(loc="upper left", fontsize=10)
plt.show()
comparison = pd.DataFrame(comparison_rows)
comparison = comparison.set_index("Variable")
display(comparison.style.format({"RMSE": "{:.6g}"}).set_uuid("ex002_1"))